In [ ]:
from ipywidgets import interact
import ipywidgets as widgets
import imgaug.augmenters as iaa
from torchvision.transforms import Compose
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import cv2

import sys
sys.path.append('../../')

from src.features.pixelwise_loss import PixelwiseLoss

from src import MODELS_DIR, MLFLOW_TRACKING_URI, DATA_PATH
from src.data import TrainValTestSplitter, MURASubset
from src.data.transforms import GrayScale, Resize, HistEqualisation, MinMaxNormalization, ToTensor
from src.features.augmentation import Augmentation
from src.models.autoencoders import BottleneckAutoencoder, BaselineAutoencoder, SkipConnection
from src.models.autoencoders import MaskedMSELoss

%matplotlib inline

In [ ]:
run_params = {
    'image_resolution': (512, 512),
    'pipeline': {
        'hist_equalisation': False,
        'data_source': 'XR_HAND_PHOTOSHOP',
    }
}

augmentation_seq = iaa.Sequential([iaa.Affine(fit_output=False, rotate=(0), order=[0, 1]),  
                                   iaa.PadToFixedSize(*run_params['image_resolution'], position='center')])

composed_transforms = Compose([GrayScale(),
                               HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                               Resize(run_params['image_resolution'], keep_aspect_ratio=True),
                               Augmentation(augmentation_seq),
                               MinMaxNormalization(),
                               ToTensor()])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.load("/home/ubuntu/mlruns/1/5ca7f67c33674926a00590752c877fe5/artifacts/BaselineAutoencoder.pth", map_location=device)
# model = torch.load("/home/ubuntu/mlruns/2/3766754ed32043bcbdc61d053615af6d/artifacts//BottleneckAutoencoder.pth", map_location=device)
# model = torch.load("/home/diana/xray/mlruns/1/28737a0f69dd4016a055433176df6a87/artifacts/VAE/data/model.pth", map_location=device)

# set loss function
outer_loss = nn.MSELoss(reduction='none')
model.eval().to(device)

In [ ]:
# Loading image
image_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}/patient00141/study1_negative/image2_cropped_1.png'

validation = MURASubset(filenames=[image_path], true_labels=[0],
                        patients=['00141'], transform=composed_transforms)
val_loader = DataLoader(validation, batch_size=1, shuffle=True, num_workers=2)

label = 0
patient = '00141'

evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=MaskedMSELoss(reduction="none"), masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data=val_loader)

for batch in validation:
    inp_image = batch['image'].cpu()[0]
loss = loss_dict['loss'][0][0]
loss = cv2.resize(loss, (512, 512))
inp_image = cv2.resize(inp_image.numpy(), (512, 512))


In [ ]:
def plot_pixelwise(max_loss):
    evaluation.add_heatmap(inp_image, label, patient, loss, original_path=None, max_loss=max_loss, 
                           sigma=4, path=None, save=False, display=True, figsize=(5, 5))
interact(plot_pixelwise, 
         max_loss = widgets.FloatSlider(min=0.0001, max=0.005, step=0.0001, value = 0.002, readout_format ='.5f'))

In [ ]:
# Loading image
image_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}/patient09769/study1_positive/image2_undetected_1.png'

validation = MURASubset(filenames=[image_path], true_labels=[1],
                        patients=['09769'], transform=composed_transforms)
val_loader = DataLoader(validation, batch_size=1, shuffle=True, num_workers=2)

label = 1
patient = '09769'

evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=MaskedMSELoss(reduction="none"), masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data=val_loader)

for batch in validation:
    inp_image = batch['image'].cpu()[0]
loss = loss_dict['loss'][0][0]
loss = cv2.resize(loss, (512, 512))
inp_image = cv2.resize(inp_image.numpy(), (512, 512))


In [ ]:
def plot_pixelwise(max_loss):
    evaluation.add_heatmap(inp_image, label, patient, loss, original_path=None, max_loss=max_loss, 
                           sigma=4, path=None, save=False, display=True, figsize=(5, 5))
interact(plot_pixelwise, 
         max_loss = widgets.FloatSlider(min=0.0001, max=0.01, step=0.0005, value = 0.001, readout_format ='.5f'))